In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import tempfile
from openai import OpenAI
import base64
from PIL import Image
import io
import os 
from typing import List, Tuple, Dict
import torch
import cv2
from ultralytics import YOLO
from matplotlib import colors
from matplotlib import patches
from FoodDetection.config import PREDICT_ARGS
from FoodDetection.food_detection import load_and_resize_image, detect_objects, plot_results
from FoodSegmentation.segmentation import LoadSAMPredictor,run_sam_with_multiple_points
from MenuSearch.search_index import FoodImageDataset,load_labels,load_faiss_index,clip_transform
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
yolo_path = r'C:\Users\user\Documents\code\korean_food_detection\tools\best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# index_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\tools\faiss_index_cheat.index"
# labels_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\tools\labels_cheat.npy"
sam_checkpoint = r"C:\Users\user\Documents\code\korean_food_detection\tools\sam_vit_b_01ec64.pth"

model_type = "vit_b"
yolo_model = YOLO(yolo_path)
sam_predictor = LoadSAMPredictor(sam_checkpoint, model_type, device='cuda', return_sam=False)
api_key=""

In [28]:
import asyncio
import base64
import io
from typing import Tuple
import numpy as np
from PIL import Image
import openai

import base64
import io
from typing import Tuple
import numpy as np
from PIL import Image
import openai
import requests
class OpenAIImageClassifier:
    def __init__(self, api_key: str):
        self.client = OpenAI(api_key=api_key)

    def encode_image_array(self, image_array: np.ndarray, target_size: Tuple[int, int] = (128, 128)) -> str:
        """Convert numpy array to base64 string after resizing."""
        image = Image.fromarray(image_array).resize(target_size, Image.LANCZOS)
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG", quality=85)
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

    def classify_image(self, image_path: str) -> str:
        """
        Classify an image from a local file path using OpenAI Vision API.
        """
        try:
            # Convert the image to a base64 string
            base64_image = self.encode_image_array(image_path)

            if not base64_image:
                return "unknown"

            # OpenAI API call
            response = self.client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": "What is in the image",
                    },
                    {
                        "role": "user",
                        "content": f"data:image/jpeg;base64,{base64_image}"
                    }
                ],
                max_tokens=50
            )

            # Extract and return the response content
            return response.choices[0].message.content.strip().lower()

        except Exception as e:
            print(f"Classification error: {e}")
            return "unknown"
# Example usage
if __name__ == "__main__":
    # Example numpy array (replace with an actual image array)
    image_url = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_exemples\1-a.jpg"

    # Initialize classifier
    classifier = OpenAIImageClassifier(api_key=api_key)

    # Classify image
    result = classifier.classify_image(image_url)
    print(f"Classification result: {result}")



Classification error: 'str' object has no attribute '__array_interface__'
Classification result: unknown


In [ ]:
import base64
import io
from PIL import Image
import openai
import numpy as np
from typing import Tuple

class OpenAIClassifier:
    def __init__(self, api_key: str):
        """Initialize OpenAI API client."""
        openai.api_key = api_key
    
    def encode_image_path(self, image_path: str, target_size: Tuple[int, int] = (128, 128)) -> str:
        """Convert an image file at a given path to a base64 string after resizing."""
        # Open the image from the local path
        image = Image.open(image_path).resize(target_size, Image.LANCZOS)
        
        # Convert image to bytes and encode as base64
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG", quality=85)
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

    def classify_image(self, image_path: str) -> str:
        """Classify image using OpenAI API by sending the base64-encoded image."""
        try:
            # Convert image to base64
            base64_image = self.encode_image_path(image_path)

            # Call OpenAI API for image classification
            response = openai.Image.create(
                model="image-classification-001",  # Assuming you have access to this model
                image=base64_image,  # Sending the base64-encoded image
                purpose="classify"
            )

            # Extract and return the result from OpenAI response
            return response["data"][0]["name"]  # Adjust based on your response structure
            
        except Exception as e:
            print(f"Classification error: {str(e)}")
            return "unknown"


# Example usage
if __name__ == "__main__":
    # Initialize classifier
    api_key = "your-api-key"  # Replace with your OpenAI API key
    classifier = OpenAIClassifier(api_key=api_key)

    # Path to a local image (replace with your image path)
    image_path = "/path/to/your/image.jpg"  # Replace with the actual file path
    
    # Classify image
    result = classifier.classify_image(image_path)
    print(f"Classification result: {result}")

